# Tune CNN: learning_rate x batch_size (decade grid)

Exhaustive grid search (Optuna `GridSampler`, seed 42) over
`learning_rate ∈ {0.01, 0.001, 0.0001}` × `batch_size ∈ {16, 32, 64}` — 9 configs,
covering three orders of magnitude. Architecture frozen (Table 3.5). Split and init
are fixed — the winner's checkpoint reuses `cnn-latest/results/splits/clean-seed42.csv`
and seed 42; CV folds reuse `cnn-latest/results/splits/clean-fold{0..4}.csv`.

This replaces an earlier log-uniform random search (archived under
`results/_archive-random-search/`): that search saturated in-domain (9/10 configs
reached ~1.0 CV accuracy, differing only in the 5th decimal of loss) and never
tested whether the search range itself was right. Discrete decades give the
examiner the same search style used for the SVM's `C`/`gamma`
(`trial.suggest_categorical` over powers of ten, `svm-tuning.ipynb`), and bracket
the current default `lr=0.001` with a decade either side. `lr=0.01` is expected to
be unstable — that establishes the boundary rather than wasting a trial.

Selection: **5-fold CV on `training-clean.npz` only** (mean test accuracy
descending, mean test loss ascending tie-break) — mirrors `svm-tuning.ipynb`'s
`StratifiedKFold(n_splits=5)` objective exactly, zero OOD contact. Only the winning
config is trained on the full 80/10/10 split; the other 8 never get a persisted
checkpoint (fit-scored-discarded, matching `svm-tuning.ipynb`).

## Config

In [1]:
from pathlib import Path
import itertools
import subprocess
import pandas as pd
from IPython.display import display, Markdown

HERE = Path.cwd()
if HERE.name != "cnn-revised":
    HERE = Path("training/notebooks/cnn-revised").resolve()
TRAINING_ROOT = HERE.parents[1]
CNN_LATEST = HERE.parent / "cnn-latest"
CV_SCORES_CSV = HERE / "results" / "cv_scores.csv"
TRIALS_CSV = HERE / "results" / "tuning_trials.csv"
FOLDS = list(range(5))

GRID_LR = [0.01, 0.001, 0.0001]
GRID_BATCH = [16, 32, 64]
CONFIGS = [
    {"trial": i, "lr": lr, "batch_size": bs}
    for i, (lr, bs) in enumerate(itertools.product(GRID_LR, GRID_BATCH))
]

cfg = pd.DataFrame([
    {"key": "here", "value": str(HERE)},
    {"key": "features", "value": str(TRAINING_ROOT / "features" / "training-clean.npz")},
    {"key": "grid", "value": f"lr {GRID_LR} x batch_size {GRID_BATCH} = {len(CONFIGS)} configs"},
    {"key": "folds", "value": FOLDS},
    {"key": "CV jobs", "value": f"{len(CONFIGS)} x {len(FOLDS)} = {len(CONFIGS) * len(FOLDS)}"},
    {"key": "fold splits (reused)", "value": str(CNN_LATEST / "results" / "splits") + "/clean-fold{k}.csv"},
    {"key": "winner split (reused)", "value": str(CNN_LATEST / "results" / "splits" / "clean-seed42.csv")},
    {"key": "fixed", "value": "es_patience=6, rlr_patience=2, rlr_factor=0.1, seed=42"},
])
display(cfg)
display(pd.DataFrame(CONFIGS))
assert all((CNN_LATEST / "results" / "splits" / f"clean-fold{f}.csv").is_file() for f in FOLDS)
assert (HERE / "run_cv.py").is_file() and (HERE / "run_tune.py").is_file()

,key,value
0,here,/home/seya/code/chord-detection/training/noteb...
1,features,/home/seya/code/chord-detection/training/featu...
2,grid,"lr [0.01, 0.001, 0.0001] x batch_size [16, 32,..."
3,folds,"[0, 1, 2, 3, 4]"
4,CV jobs,9 x 5 = 45
5,fold splits (reused),/home/seya/code/chord-detection/training/noteb...
6,winner split (reused),/home/seya/code/chord-detection/training/noteb...
7,fixed,"es_patience=6, rlr_patience=2, rlr_factor=0.1,..."


,trial,lr,batch_size
0,0,0.0100,16
1,1,0.0100,32
2,2,0.0100,64
3,3,0.0010,16
4,4,0.0010,32
5,5,0.0010,64
6,6,0.0001,16
7,7,0.0001,32
8,8,0.0001,64


## Grid search — 9 configs x 5-fold CV (45 runs)

Exhaustive, not random sampling — every `(lr, batch_size)` combination is scored.
Each `(config, fold)` job is an isolated OS process (`run_cv.py`), GPU memory
released on exit; per-fold models are fit, scored, discarded. Resume-friendly: a
`(trial, fold)` already in `cv_scores.csv` is skipped.

In [2]:
jobs = [(c["trial"], f, c["lr"], c["batch_size"]) for c in CONFIGS for f in FOLDS]
done = pd.read_csv(CV_SCORES_CSV) if CV_SCORES_CSV.exists() else pd.DataFrame(columns=["trial", "fold"])
pending = [
    (t, f, lr, bs) for t, f, lr, bs in jobs
    if done.empty or not ((done.trial == t) & (done.fold == f)).any()
]
display(Markdown(f"Pending **{len(pending)}** / {len(jobs)}"))

for trial, fold, lr, bs in pending:
    display(Markdown(f"### config {trial} (lr={lr}, batch_size={bs}) fold {fold}"))
    result = subprocess.run(
        [
            "uv", "run", "python", "run_cv.py",
            "--trial", str(trial),
            "--fold", str(fold),
            "--lr", str(lr),
            "--batch-size", str(bs),
        ],
        cwd=HERE,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"config{trial}/fold{fold} exited {result.returncode}")

display(Markdown("All jobs done." if not pending else "Ran all pending jobs."))

Pending **0** / 45

All jobs done.

## CV scores per config

In [3]:
cv = pd.read_csv(CV_SCORES_CSV)
assert len(cv) == len(CONFIGS) * len(FOLDS), f"expected {len(CONFIGS) * len(FOLDS)} rows, got {len(cv)}"

agg = cv.groupby("trial").agg(
    lr=("lr", "first"),
    batch_size=("batch_size", "first"),
    mean_test_accuracy=("test_accuracy", "mean"),
    std_test_accuracy=("test_accuracy", "std"),
    mean_test_loss=("test_loss", "mean"),
    std_test_loss=("test_loss", "std"),
    mean_val_loss=("val_loss", "mean"),
).reset_index()
agg = agg.sort_values(
    ["mean_test_accuracy", "mean_test_loss"], ascending=[False, True]
).reset_index(drop=True)

winner_trial = int(agg.iloc[0].trial)
winner_cfg = agg.iloc[0]

show = agg.copy()
show.insert(0, "", ["*" if t == winner_trial else "" for t in show.trial])

display(Markdown(
    f"**Winner: config {winner_trial}** — lr={winner_cfg.lr:g}, "
    f"batch_size={int(winner_cfg.batch_size)}, "
    f"mean_test_accuracy={winner_cfg.mean_test_accuracy:.4f} "
    f"(± {winner_cfg.std_test_accuracy:.4f}), "
    f"mean_test_loss={winner_cfg.mean_test_loss:.6f} "
    f"(accuracy desc, loss asc tie-break). `*` marks it."
))
display(show.round(6))

n_diverged = int((agg.mean_test_accuracy < 0.9).sum())
if n_diverged:
    bad = agg.loc[agg.mean_test_accuracy < 0.9]
    display(Markdown(
        f"**{n_diverged} config(s) collapsed under CV** (mean accuracy < 0.9): "
        + "; ".join(
            f"lr={r.lr:g}/batch={int(r.batch_size)} (acc={r.mean_test_accuracy:.3f} "
            f"± {r.std_test_accuracy:.3f})"
            for _, r in bad.iterrows()
        )
        + " — this establishes the instability boundary rather than indicating a search error."
    ))

**Winner: config 6** — lr=0.0001, batch_size=16, mean_test_accuracy=1.0000 (± 0.0000), mean_test_loss=0.000103 (accuracy desc, loss asc tie-break). `*` marks it.

,,trial,lr,batch_size,mean_test_accuracy,std_test_accuracy,mean_test_loss,std_test_loss,mean_val_loss
0,*,6,0.0001,16,1.000000,0.000000,0.000103,0.000169,0.000138
1,,3,0.0010,16,1.000000,0.000000,0.000231,0.000387,0.001099
2,,7,0.0001,32,0.999861,0.000312,0.000294,0.000538,0.000291
3,,4,0.0010,32,0.999861,0.000312,0.000373,0.000602,0.000553
4,,5,0.0010,64,0.999861,0.000312,0.000606,0.001086,0.001440
5,,8,0.0001,64,0.999861,0.000312,0.000624,0.001361,0.001008
6,,0,0.0100,16,0.222172,0.434430,2.867125,1.601693,2.868277
7,,2,0.0100,64,0.027886,0.000011,3.583460,0.000017,3.583542
8,,1,0.0100,32,0.027886,0.000011,3.583532,0.000069,3.583622


**3 config(s) collapsed under CV** (mean accuracy < 0.9): lr=0.01/batch=16 (acc=0.222 ± 0.434); lr=0.01/batch=64 (acc=0.028 ± 0.000); lr=0.01/batch=32 (acc=0.028 ± 0.000) — this establishes the instability boundary rather than indicating a search error.

## Train the winner (full 80/10/10 split)

Only the winning config gets a real checkpoint. `run_tune.py` trains it once on
the fixed `clean-seed42` 80/10/10 split (`cnn-latest/results/splits/clean-seed42.csv`)
and also reports OOD accuracy on A1–B2 as a built-in sanity check — that read
happens *after* selection is already final on CV alone, so it carries no leak,
but `run_eval.py` (next notebook) remains the authoritative source for reported
numbers, since it also covers `vivo` and the overlay sets.

In [4]:
winner_lr = float(winner_cfg.lr)
winner_batch = int(winner_cfg.batch_size)
hist_path = HERE / "results" / "history" / f"trial{winner_trial:02d}.csv"

if not hist_path.exists():
    result = subprocess.run(
        [
            "uv", "run", "python", "run_tune.py",
            "--trial", str(winner_trial),
            "--lr", str(winner_lr),
            "--batch-size", str(winner_batch),
        ],
        cwd=HERE,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"winner training exited {result.returncode}")
else:
    display(Markdown(f"Checkpoint for config {winner_trial} already trained, skipping."))

row = pd.read_csv(TRIALS_CSV).loc[lambda d: d.trial == winner_trial].iloc[0]
display(Markdown(
    f"Trained. In-domain test_accuracy={row.test_accuracy:.4f}. "
    f"OOD sanity check — A1={row.A1:.4f} A2={row.A2:.4f} B1={row.B1:.4f} B2={row.B2:.4f} "
    f"(sanity only; `run_eval.py` is authoritative)."
))

Checkpoint for config 6 already trained, skipping.

Trained. In-domain test_accuracy=1.0000. OOD sanity check — A1=1.0000 A2=1.0000 B1=1.0000 B2=1.0000 (sanity only; `run_eval.py` is authoritative).

## Promote the winner

Copies `weights/trial{winner}.keras` → `weights/clean-seed42.keras` and the
matching history CSV, the filename every downstream consumer
(`run_eval.py`, `onset-classify.ipynb`, `app/models` symlink) expects.

In [5]:
import shutil

src_w = HERE / "weights" / f"trial{winner_trial:02d}.keras"
dst_w = HERE / "weights" / "clean-seed42.keras"
src_h = HERE / "results" / "history" / f"trial{winner_trial:02d}.csv"
dst_h = HERE / "results" / "history" / "clean-seed42.csv"

assert src_w.is_file(), src_w
shutil.copy2(src_w, dst_w)
shutil.copy2(src_h, dst_h)

display(Markdown(
    f"Promoted trial {winner_trial} → `{dst_w.relative_to(TRAINING_ROOT)}` "
    f"({dst_w.stat().st_size / 1e6:.1f} MB)"
))

Promoted trial 6 → `notebooks/cnn-revised/weights/clean-seed42.keras` (124.2 MB)